In [1]:
import pandas as pd

df = pd.read_csv('../data/db/events.csv')

df.head()

,id,date_start,date_end,event
0,00dc6acf-fc00-482a-968c-7cce8d4f5438,2000-01-01,NaN,Деноминация белорусского рубля;
1,00dc6acf-fc00-4057-afdb-327b500359f6,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н..."
2,00dc6acf-fc00-465f-8ed3-e93302fd57b9,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог..."
3,00dc6ff6-7800-4b0b-8f50-960bb06325fa,2000-01-02,NaN,крушение украинского сухогруза типа «река-море...
4,00dc751c-7400-42a6-b4e7-ed2bd30dc331,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...


In [2]:
import spacy
from spacy.matcher import DependencyMatcher

nlp = spacy.load("ru_core_news_lg")
dm = DependencyMatcher(nlp.vocab)

# 1️⃣ Ввели / наложили санкции
pattern_action = [
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"LEMMA": {"IN": ["ввести", "наложить"]}}},
    {"LEFT_ID": "verb", "REL_OP": ">>", "RIGHT_ID": "sanction", "RIGHT_ATTRS": {"LEMMA": "санкция"}},
]

lemmas_pkg = [
    "принять", "принят", "принятие",
    "утвердить", "утвержден", "утверждение",
    "одобрить", "одобрен", "одобрение"
]

pattern_pkg_any = [
    {
        "RIGHT_ID": "root",
        "RIGHT_ATTRS": {
            "LEMMA": {"IN": lemmas_pkg}
        },
    },
    {
        "LEFT_ID": "root",
        "REL_OP": ">>",  # допускаем промежуточные звенья (числительные, прилагательные)
        "RIGHT_ID": "package",
        "RIGHT_ATTRS": {"LEMMA": "пакет"},
    },
    {
        "LEFT_ID": "package",
        "REL_OP": ">>",
        "RIGHT_ID": "sanction",
        "RIGHT_ATTRS": {"LEMMA": "санкция"},
    },
]

dm.add("SANCTION", [pattern_action, pattern_pkg_any])

# === Тест ===
text = """
Принят 9 пакет санкций против России.
"""

doc = nlp(text)

matches = dm(doc)
seen = set()
for mid, toks in matches:
    span = doc[min(toks): max(toks) + 1]
    key = tuple(sorted(toks))
    if key in seen:
        continue
    seen.add(key)
    print(f"{nlp.vocab.strings[mid]} → {span.text}")

SANCTION → Принят 9 пакет санкций


In [3]:
def has_sanction(text, matcher=dm):
    doc = nlp(text)
    matches = matcher(doc)
    return len(matches) > 0


df["is_sanction"] = df["event"].apply(has_sanction)

print(len(df[df["is_sanction"] == True]))
df[df["is_sanction"] == True]

20


,id,date_start,date_end,event,is_sanction
1560,010b3557-f400-4ade-9d7d-9e5e0558d163,2006-05-15,NaN,США ввели санкции против Венесуэлы «за недоста...,True
1733,0115d475-7400-49e8-bc69-cf1695cd9f16,2007-10-25,NaN,США ввели дополнительные финансовые санкции пр...,True
4657,017f1eba-7c00-4a09-b0ae-7fb048f2376a,2022-02-21,2022-02-23,Принятие первого пакета санкций против России.,True
4660,017f2b9a-7200-4e6e-bd82-9da51f575bd3,2022-02-24,2022-02-25,Принятие второго пакета санкций против России.,True
4665,017f5c86-fc00-4bda-b58f-dc6b1ea4dc81,2022-02-26,2022-03-14,Принятие третьего пакета санкций против России.,True
4680,017fbe5f-7000-43f6-b297-6ac26c2b21f3,2022-03-15,2022-04-04,Принятие четвертого пакета санкций против России.,True
4696,01808c5d-f000-4a32-b589-5e71ec54e94b,2022-04-05,2022-06-02,Принятие пятого пакета санкций против России.,True
4728,01819302-7400-4b2a-966f-0b2829f35a0a,2022-06-03,2022-07-15,Принятие шестого пакета санкций против России.,True
4764,0182df2c-7200-4454-9b9f-fc96608eb0ce,2022-07-21,2022-10-04,Принятие седьмого пакета санкций против России.,True
4820,01845ed6-7800-48d7-bd6a-3dece3d0483b,2022-10-06,2022-12-15,Принятие 8 пакета санкций против России.,True


In [4]:
import pandas as pd
import os

def tag_sanction_events(df, event_tags_path: str):
    """
    Добавляет тэг SANCTIONS к событиям, где has_sanction == True.
    Перед выполнением запрашивает подтверждение в консоли.

    df — DataFrame с событиями (обязательно с колонками ['id', 'event', 'is_sanction'])
    event_tags_path — путь к файлу event_tags.csv
    """

    confirm = input(f"⚠️ Добавить тэг 'SANCTIONS' для событий с санкциями в '{event_tags_path}'? [y/n]: ").strip().lower()
    if confirm != "y":
        print("Операция отменена пользователем.")
        return

    sanctions_tag_code = "SANCTIONS"

    # === 🔹 Фильтруем события, где есть санкции ===
    sanction_events = df[df["is_sanction"] == True][["id"]].copy()

    if sanction_events.empty:
        print("⚠️ Не найдено событий с санкциями.")
        return

    sanction_events["tag_code"] = sanctions_tag_code

    # === 🔹 Загружаем или создаём event_tags.csv ===
    if os.path.exists(event_tags_path):
        event_tags = pd.read_csv(event_tags_path, encoding="utf-8-sig")
    else:
        print(f'Файла {event_tags_path} не существует. Создайте его и повторите попытку.')
        return

    # === 🔹 Добавляем новые связи ===
    new_records = pd.DataFrame({
        "event_id": sanction_events["id"],
        "tag_code": sanction_events["tag_code"]
    })

    # Удаляем дубликаты (если запись уже есть)
    combined = pd.concat([event_tags, new_records], ignore_index=True)
    combined = combined.drop_duplicates(subset=["event_id", "tag_code"])

    # === 🔹 Сохраняем обновлённый файл ===
    combined.to_csv(event_tags_path, index=False, encoding="utf-8-sig")

    print(f"✅ Добавлено/обновлено {len(new_records)} связей в '{event_tags_path}'.")

tag_sanction_events(df, '../data/db/event_tags.csv')

✅ Добавлено/обновлено 20 связей в '../data/db/event_tags.csv'.
